# Qwen Teacher T4 Experiment Notebook

Stable Kaggle single-T4 notebook for usable-teacher selection.

Run order:
1. Setup
2. Config
3. Input validation
4. Stage 1 prompt screen
5. Stage 1 inspection
6. Stage 2 model selection
7. Stage 2 inspection
8. Stage 3 holdout
9. Full run
10. Final inspection

Notes:
- Use a single `NvidiaTeslaT4` runtime.
- The notebook embeds `teacher_runtime.py`, so it does not need sibling repo files at runtime.
- Stage winners are chosen by usable yield and gold-backed fidelity metrics, not exact parse-valid alone.
- Leave `ALLOW_FULL_RUN = False` until the holdout stage passes.



In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import sys

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

EMBEDDED_RUNTIME_SOURCE = 'from __future__ import annotations\n\nimport gc\nimport json\nimport math\nimport os\nimport re\nimport time\nfrom collections import Counter\nfrom dataclasses import asdict, dataclass, field\nfrom difflib import SequenceMatcher\nfrom pathlib import Path\nfrom statistics import mean, median\nfrom typing import Any\n\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\n\nDEFAULT_MODEL = "Qwen/Qwen3-4B-Instruct-2507"\nDEFAULT_STAGE1_MODEL = "Qwen/Qwen3-1.7B"\nDEFAULT_STAGE2_MODELS = (\n    "Qwen/Qwen3-1.7B",\n    "Qwen/Qwen3-4B-Instruct-2507",\n)\nPROMPT_METHODS = (\n    "baseline",\n    "strict_template",\n    "heuristic_rewrite",\n)\nDEFAULT_PROMPT_METHOD = "strict_template"\nDEFAULT_TRANSCRIPT_TURN_CHAR_LIMIT = 1200\nDEFAULT_TRANSCRIPT_CHAR_BUDGET = 14000\nDEFAULT_DEBUG_TEXT_CHAR_LIMIT = 6000\nDEFAULT_SMOKE_LIMIT = 10\n\nPREAMBLE = "We are continuing an earlier conversation, not starting fresh."\nTHINK_BLOCK_PATTERN = re.compile(r"<think>([\\s\\S]*?)</think>", re.IGNORECASE)\nREQUIRED_SECTIONS = [\n    PREAMBLE,\n    "Objective:",\n    "Established context:",\n    "Decisions already made:",\n    "Rejected paths / do not revisit unless necessary:",\n    "Open questions:",\n    "Where we left off:",\n    "Continue from this exact state.",\n    "My next request:",\n]\nCONTENT_SECTION_HEADERS = REQUIRED_SECTIONS[1:]\nSECTION_LINE_PATTERNS = {\n    PREAMBLE: re.compile(rf"^{re.escape(PREAMBLE)}$", re.MULTILINE),\n    "Objective:": re.compile(r"^Objective:$", re.MULTILINE),\n    "Established context:": re.compile(r"^Established context:$", re.MULTILINE),\n    "Decisions already made:": re.compile(r"^Decisions already made:$", re.MULTILINE),\n    "Rejected paths / do not revisit unless necessary:": re.compile(\n        r"^Rejected paths / do not revisit unless necessary:$",\n        re.MULTILINE,\n    ),\n    "Open questions:": re.compile(r"^Open questions:$", re.MULTILINE),\n    "Where we left off:": re.compile(r"^Where we left off:$", re.MULTILINE),\n    "Continue from this exact state.": re.compile(r"^Continue from this exact state\\.$", re.MULTILINE),\n    "My next request:": re.compile(r"^My next request:$", re.MULTILINE),\n}\nPROMPT_ECHO_MARKERS = (\n    "\\nRequirements:\\n",\n    "\\nKey requirements:\\n",\n    "Exactly the structure and section order:",\n    "Rewrite the supplied conversation state into the exact brief template below.",\n    "Return exactly this structure:",\n    "\\nSource transcript:\\n",\n    "Do not invent goals, decisions, constraints, files, APIs, or next steps.",\n    "We are given a transcript",\n    "We have the rolling state helper provided",\n    "We have the transcript from the conversation",\n    "Let\'s break down",\n)\nPLACEHOLDER_BULLET_PATTERN = re.compile(r"(?m)^-\\s*\\.\\.\\.\\s*$")\nINLINE_SECTION_PATTERNS = {\n    "Objective:": re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Objective\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"),\n    "Established context:": re.compile(\n        r"(?mi)^\\s*(?:\\*\\*)?\\s*Established context\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"\n    ),\n    "Decisions already made:": re.compile(\n        r"(?mi)^\\s*(?:\\*\\*)?\\s*Decisions already made\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"\n    ),\n    "Rejected paths / do not revisit unless necessary:": re.compile(\n        r"(?mi)^\\s*(?:\\*\\*)?\\s*Rejected paths / do not revisit unless necessary\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"\n    ),\n    "Open questions:": re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Open questions\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"),\n    "Where we left off:": re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Where we left off\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"),\n    "My next request:": re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*My next request\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"),\n}\nSECTION_HEADER_NORMALIZATIONS = (\n    (re.compile(rf"(?mi)^\\s*{re.escape(PREAMBLE)}\\s*$"), PREAMBLE),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Objective\\s*(?:\\*\\*)?\\s*:\\s*$"), "Objective:"),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Established context\\s*(?:\\*\\*)?\\s*:\\s*$"), "Established context:"),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Decisions already made\\s*(?:\\*\\*)?\\s*:\\s*$"), "Decisions already made:"),\n    (\n        re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Rejected paths / do not revisit unless necessary\\s*(?:\\*\\*)?\\s*:\\s*$"),\n        "Rejected paths / do not revisit unless necessary:",\n    ),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Open questions\\s*(?:\\*\\*)?\\s*:\\s*$"), "Open questions:"),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Where we left off\\s*(?:\\*\\*)?\\s*:\\s*$"), "Where we left off:"),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Continue from this exact state\\.?\\s*$"), "Continue from this exact state."),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*My next request\\s*(?:\\*\\*)?\\s*:\\s*$"), "My next request:"),\n)\nKNOWN_INPUT_PATHS = {\n    "full": "/kaggle/input/qwen4b-teacher-2gpu-inputs/teacher_input_rows.json",\n    "repair": "/kaggle/input/qwen4b-teacher-2gpu-repair-inputs/live_extension_review_queue_parallel_assistant_repaired_v11.json",\n}\nSTATUS_PRIORITY = {\n    "hard_reject": 0,\n    "soft_accept": 1,\n    "format_clean": 2,\n}\nGENERIC_FALLBACK_LINES = {\n    "Objective still needs confirmation.",\n    "No explicit constraints have been captured yet.",\n    "No firm decisions are recorded yet.",\n    "No rejected paths have been recorded yet.",\n    "No open questions are active right now.",\n    "Resume from the latest visible turn.",\n}\nSECTION_KEYS = (\n    "objective",\n    "constraints",\n    "decisions",\n    "rejected",\n    "open_questions",\n    "next_step",\n)\nSECTION_WEIGHTS = {\n    "objective": 0.20,\n    "constraints": 0.15,\n    "decisions": 0.20,\n    "rejected": 0.10,\n    "open_questions": 0.10,\n    "next_step": 0.25,\n}\nACTIONABLE_REINTRO_SECTIONS = ("objective", "decisions", "next_step")\nSTATE_MATCH_THRESHOLD = 0.58\nSECTION_LABELS = {\n    "objective": "Objective:",\n    "constraints": "Established context:",\n    "decisions": "Decisions already made:",\n    "rejected": "Rejected paths / do not revisit unless necessary:",\n    "open_questions": "Open questions:",\n    "next_step": "Where we left off:",\n}\nSCALAR_METRIC_NAMES = (\n    "state_fidelity_score",\n    "teacher_similarity_score",\n    "must_include_recall",\n    "avoid_hit_rate",\n    "rejected_path_reintroduction_rate",\n    "carry_forward_accuracy",\n    "unwarranted_change_rate",\n    "next_step_fidelity",\n    "brevity_readability_score",\n    "hallucination_penalty",\n)\n\n\n@dataclass\nclass BriefSections:\n    objective: str\n    constraints: list[str] = field(default_factory=list)\n    decisions: list[str] = field(default_factory=list)\n    rejected: list[str] = field(default_factory=list)\n    open_questions: list[str] = field(default_factory=list)\n    next_step: str = ""\n\n    @classmethod\n    def from_dict(cls, raw: dict[str, Any]) -> "BriefSections":\n        return cls(\n            objective=str(raw.get("objective", "")).strip(),\n            constraints=[str(item).strip() for item in raw.get("constraints", []) if str(item).strip()],\n            decisions=[str(item).strip() for item in raw.get("decisions", []) if str(item).strip()],\n            rejected=[str(item).strip() for item in raw.get("rejected", []) if str(item).strip()],\n            open_questions=[\n                str(item).strip()\n                for item in raw.get("open_questions", raw.get("openQuestions", []))\n                if str(item).strip()\n            ],\n            next_step=str(raw.get("next_step", raw.get("nextStep", ""))).strip(),\n        )\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n    def section_items(self) -> dict[str, list[str]]:\n        return {\n            "objective": [self.objective] if self.objective else [],\n            "constraints": list(self.constraints),\n            "decisions": list(self.decisions),\n            "rejected": list(self.rejected),\n            "open_questions": list(self.open_questions),\n            "next_step": [self.next_step] if self.next_step else [],\n        }\n\n\n@dataclass\nclass ExperimentMetrics:\n    state_fidelity_score: float\n    teacher_similarity_score: float\n    must_include_recall: float\n    avoid_hit_rate: float\n    rejected_path_reintroduction_rate: float\n    carry_forward_accuracy: float\n    unwarranted_change_rate: float\n    next_step_fidelity: float\n    brevity_readability_score: float\n    hallucination_penalty: float\n    section_scores: dict[str, float]\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\n@dataclass\nclass GenerationOutcome:\n    brief: str\n    auxiliary_rationale: str\n    status: str\n    issues: list[str]\n    debug_payload: dict[str, Any]\n    duration_ms: int\n\n\ndef ensure_parent(path: str | None) -> None:\n    if path:\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n\n\ndef write_json(path: str | Path, payload: Any) -> str:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, indent=2) + "\\n", encoding="utf-8")\n    return str(path)\n\n\ndef read_json(path: str | Path) -> Any:\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef sanitize_text(text: Any) -> str:\n    return re.sub(r"\\s+", " ", str(text or "")).strip()\n\n\ndef trim_text(text: Any, max_chars: int) -> str:\n    compact = sanitize_text(text)\n    if len(compact) <= max_chars:\n        return compact\n    return f"{compact[:max_chars].strip()}..."\n\n\ndef trim_debug_text(text: Any, max_chars: int = DEFAULT_DEBUG_TEXT_CHAR_LIMIT) -> str:\n    value = str(text or "")\n    if len(value) <= max_chars:\n        return value\n    remaining = len(value) - max_chars\n    return f"{value[:max_chars].rstrip()}\\n...[truncated {remaining} chars]"\n\n\ndef slugify(value: str) -> str:\n    lowered = value.lower().strip()\n    lowered = re.sub(r"[^a-z0-9]+", "-", lowered)\n    return lowered.strip("-") or "value"\n\n\ndef safe_ratio(numerator: int, denominator: int) -> float:\n    if denominator <= 0:\n        return 0.0\n    return numerator / denominator\n\n\ndef safe_mean(values: list[float]) -> float:\n    if not values:\n        return 0.0\n    return mean(values)\n\n\ndef normalize_text(value: str) -> str:\n    return re.sub(r"\\s+", " ", value.strip().lower())\n\n\ndef tokenize(value: str) -> list[str]:\n    return [token for token in re.findall(r"[a-z0-9\']+", normalize_text(value)) if token]\n\n\nSTOP_WORDS = {\n    "a",\n    "an",\n    "and",\n    "are",\n    "as",\n    "at",\n    "be",\n    "by",\n    "for",\n    "from",\n    "how",\n    "if",\n    "in",\n    "into",\n    "is",\n    "it",\n    "of",\n    "on",\n    "or",\n    "that",\n    "the",\n    "this",\n    "to",\n    "we",\n    "with",\n    "you",\n}\n\n\ndef content_tokens(value: str) -> list[str]:\n    return [token for token in tokenize(value) if token not in STOP_WORDS]\n\n\ndef unique_content_tokens(value: str) -> set[str]:\n    return set(content_tokens(value))\n\n\ndef jaccard_similarity(left: str, right: str) -> float:\n    left_tokens = unique_content_tokens(left)\n    right_tokens = unique_content_tokens(right)\n    if not left_tokens and not right_tokens:\n        return 1.0\n    if not left_tokens or not right_tokens:\n        return 0.0\n    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)\n\n\ndef sequence_similarity(left: str, right: str) -> float:\n    if not left and not right:\n        return 1.0\n    if not left or not right:\n        return 0.0\n    return SequenceMatcher(None, normalize_text(left), normalize_text(right)).ratio()\n\n\ndef text_similarity(left: str, right: str) -> float:\n    return 0.6 * jaccard_similarity(left, right) + 0.4 * sequence_similarity(left, right)\n\n\ndef novel_token_ratio(output: str, evidence: str) -> float:\n    output_tokens = unique_content_tokens(output)\n    evidence_tokens = unique_content_tokens(evidence)\n    if not output_tokens:\n        return 0.0\n    return len(output_tokens - evidence_tokens) / len(output_tokens)\n\n\ndef _clean_item(value: str) -> str:\n    cleaned = value.strip()\n    if cleaned.startswith("-"):\n        cleaned = cleaned[1:].strip()\n    return cleaned\n\n\ndef parse_continuation_brief(text: str) -> BriefSections:\n    lines = text.replace("\\r\\n", "\\n").replace("\\r", "\\n").split("\\n")\n    section_items: dict[str, list[str]] = {key: [] for key in SECTION_KEYS}\n    seen_headers: set[str] = set()\n    active_section: str | None = None\n\n    header_map = {\n        "Objective:": "objective",\n        "Established context:": "constraints",\n        "Decisions already made:": "decisions",\n        "Rejected paths / do not revisit unless necessary:": "rejected",\n        "Open questions:": "open_questions",\n        "Where we left off:": "next_step",\n    }\n\n    for raw_line in lines:\n        line = raw_line.strip()\n        if not line:\n            continue\n\n        header_key = header_map.get(line)\n        if header_key:\n            active_section = header_key\n            seen_headers.add(header_key)\n            continue\n\n        if line.startswith((PREAMBLE, "Continue from this exact state.", "My next request:")):\n            active_section = None\n            continue\n\n        if active_section is None:\n            continue\n\n        item = _clean_item(line)\n        if not item:\n            continue\n        section_items[active_section].append(item)\n\n    missing = [header for header in header_map.values() if header not in seen_headers]\n    if missing:\n        raise ValueError(f"Continuation brief is missing required sections: {\', \'.join(missing)}")\n\n    if not section_items["objective"]:\n        raise ValueError("Continuation brief did not contain an objective item.")\n    if not section_items["next_step"]:\n        raise ValueError("Continuation brief did not contain a next_step item.")\n\n    return BriefSections(\n        objective=section_items["objective"][0],\n        constraints=section_items["constraints"],\n        decisions=section_items["decisions"],\n        rejected=section_items["rejected"],\n        open_questions=section_items["open_questions"],\n        next_step=section_items["next_step"][0],\n    )\n\n\ndef _list_f1(predicted: list[str], gold: list[str]) -> float:\n    if not predicted and not gold:\n        return 1.0\n    if not predicted or not gold:\n        return 0.0\n\n    recall = mean(max(text_similarity(gold_item, predicted_item) for predicted_item in predicted) for gold_item in gold)\n    precision = mean(max(text_similarity(predicted_item, gold_item) for gold_item in gold) for predicted_item in predicted)\n    if recall + precision == 0:\n        return 0.0\n    return 2 * recall * precision / (recall + precision)\n\n\ndef score_sections(predicted: BriefSections | None, gold: BriefSections) -> dict[str, float]:\n    if predicted is None:\n        return {section: 0.0 for section in SECTION_KEYS}\n\n    predicted_items = predicted.section_items()\n    gold_items = gold.section_items()\n    return {section: _list_f1(predicted_items[section], gold_items[section]) for section in SECTION_KEYS}\n\n\ndef _state_item_matches(left: str, right: str) -> bool:\n    return text_similarity(left, right) >= STATE_MATCH_THRESHOLD\n\n\ndef _carry_forward_metrics(row: dict[str, Any], predicted: BriefSections | None) -> tuple[float, float]:\n    prior_items = BriefSections.from_dict(row.get("rolling_state") or {}).section_items()\n    gold_items = BriefSections.from_dict(row.get("gold_sections") or {}).section_items()\n    predicted_items = predicted.section_items() if predicted is not None else {section: [] for section in SECTION_KEYS}\n\n    persistent_targets: list[tuple[str, str, str]] = []\n    for section in SECTION_KEYS:\n        for prior_item in prior_items[section]:\n            matching_gold = next(\n                (gold_item for gold_item in gold_items[section] if _state_item_matches(prior_item, gold_item)),\n                None,\n            )\n            if matching_gold:\n                persistent_targets.append((section, prior_item, matching_gold))\n\n    if not persistent_targets:\n        return 1.0, 0.0\n\n    persisted = 0\n    unwarranted_changes = 0\n    for section, prior_item, gold_item in persistent_targets:\n        target_items = predicted_items[section]\n        if any(_state_item_matches(prior_item, item) or _state_item_matches(gold_item, item) for item in target_items):\n            persisted += 1\n        else:\n            unwarranted_changes += 1\n\n    total = len(persistent_targets)\n    return persisted / total, unwarranted_changes / total\n\n\ndef _rejected_path_reintroduction_rate(\n    predicted: BriefSections | None,\n    generated_brief: str,\n    expected_avoid: list[str],\n    rejected_items: list[str],\n) -> float:\n    if predicted is None:\n        actionable_items = [generated_brief]\n    else:\n        predicted_items = predicted.section_items()\n        actionable_items = [\n            item\n            for section in ACTIONABLE_REINTRO_SECTIONS\n            for item in predicted_items[section]\n        ]\n\n    actionable_text = "\\n".join(actionable_items).lower()\n    if expected_avoid:\n        hits = sum(1 for term in expected_avoid if term.lower() in actionable_text)\n        return hits / len(expected_avoid)\n\n    if not rejected_items:\n        return 0.0\n\n    hits = 0\n    for rejected_item in rejected_items:\n        if any(_state_item_matches(rejected_item, candidate) for candidate in actionable_items):\n            hits += 1\n    return hits / len(rejected_items)\n\n\ndef score_output(row: dict[str, Any], generated_brief: str, predicted_sections: BriefSections | None) -> ExperimentMetrics:\n    gold_sections = BriefSections.from_dict(row.get("gold_sections") or {})\n    section_scores = score_sections(predicted_sections, gold_sections)\n    state_fidelity = sum(section_scores[section] * SECTION_WEIGHTS[section] for section in SECTION_KEYS)\n\n    output_lower = generated_brief.lower()\n    expected_must_include = [str(item).strip() for item in row.get("expected_must_include", []) if str(item).strip()]\n    expected_avoid = [str(item).strip() for item in row.get("expected_avoid", []) if str(item).strip()]\n    must_hits = sum(1 for term in expected_must_include if term.lower() in output_lower)\n    must_recall = 1.0 if not expected_must_include else must_hits / len(expected_must_include)\n\n    carry_forward_accuracy, unwarranted_change_rate = _carry_forward_metrics(row, predicted_sections)\n    rejected_path_reintroduction_rate = _rejected_path_reintroduction_rate(\n        predicted_sections,\n        generated_brief,\n        expected_avoid,\n        gold_sections.rejected,\n    )\n    avoid_hit_rate = rejected_path_reintroduction_rate\n\n    required_headers = sum(1 for label in SECTION_LABELS.values() if label in generated_brief)\n    structure_score = required_headers / len(SECTION_LABELS)\n    length_ok = 1.0 if 250 <= len(generated_brief) <= 2200 else 0.0\n    brevity_readability = 0.5 * structure_score + 0.5 * length_ok\n\n    teacher_similarity = sequence_similarity(generated_brief, str(row.get("teacher_draft_brief", "")))\n    evidence = "\\n".join(str(turn.get("text", "")) for turn in row.get("transcript_turns", [])) + "\\n" + str(\n        row.get("teacher_draft_brief", "")\n    )\n    hallucination_penalty = max(0.0, novel_token_ratio(generated_brief, evidence) - 0.38)\n\n    return ExperimentMetrics(\n        state_fidelity_score=round(state_fidelity, 4),\n        teacher_similarity_score=round(teacher_similarity, 4),\n        must_include_recall=round(must_recall, 4),\n        avoid_hit_rate=round(avoid_hit_rate, 4),\n        rejected_path_reintroduction_rate=round(rejected_path_reintroduction_rate, 4),\n        carry_forward_accuracy=round(carry_forward_accuracy, 4),\n        unwarranted_change_rate=round(unwarranted_change_rate, 4),\n        next_step_fidelity=round(section_scores["next_step"], 4),\n        brevity_readability_score=round(brevity_readability, 4),\n        hallucination_penalty=round(hallucination_penalty, 4),\n        section_scores={section: round(score, 4) for section, score in section_scores.items()},\n    )\n\n\ndef aggregate_gold_metrics(rows: list[dict[str, Any]]) -> dict[str, Any]:\n    metric_dicts: list[ExperimentMetrics] = []\n    for row in rows:\n        if not str(row.get("gold_brief", "")).strip():\n            continue\n        generated_brief = str(row.get("teacher_draft_brief", "")).strip()\n        predicted_sections = None\n        if generated_brief:\n            try:\n                predicted_sections = parse_continuation_brief(generated_brief)\n            except Exception:\n                predicted_sections = None\n        metric_dicts.append(score_output(row, generated_brief, predicted_sections))\n\n    if not metric_dicts:\n        return {"count": 0}\n\n    payload: dict[str, Any] = {"count": len(metric_dicts)}\n    for metric_name in SCALAR_METRIC_NAMES:\n        payload[metric_name] = round(safe_mean([getattr(metrics, metric_name) for metrics in metric_dicts]), 4)\n    return payload\n\n\ndef build_transcript_text(\n    transcript_turns: list[dict[str, Any]],\n    *,\n    per_turn_char_limit: int,\n    total_char_budget: int,\n) -> str:\n    rendered_turns = [\n        f"{index + 1}. {str(turn.get(\'role\', \'\')).upper()}: {trim_text(turn.get(\'text\', \'\'), per_turn_char_limit)}"\n        for index, turn in enumerate(transcript_turns)\n    ]\n    if not rendered_turns:\n        return "No transcript turns were provided."\n\n    if total_char_budget <= 0:\n        return "\\n".join(rendered_turns)\n\n    kept_turns: list[str] = []\n    used_chars = 0\n    for turn_text in reversed(rendered_turns):\n        turn_size = len(turn_text) + 1\n        if kept_turns and used_chars + turn_size > total_char_budget:\n            break\n        kept_turns.append(turn_text)\n        used_chars += turn_size\n        if used_chars >= total_char_budget:\n            break\n\n    kept_turns.reverse()\n    omitted_turns = len(rendered_turns) - len(kept_turns)\n    if omitted_turns > 0:\n        kept_turns.insert(0, f"[{omitted_turns} earlier turns omitted to fit the transcript budget.]")\n    return "\\n".join(kept_turns)\n\n\ndef render_list(title: str, items: list[str], fallback: str) -> str:\n    lines = items if items else [fallback]\n    return f"{title}:\\n" + "\\n".join(f"- {item}" for item in lines)\n\n\ndef render_template_list(items: list[str], fallback: str) -> str:\n    lines = [item for item in items if item]\n    return "\\n".join(f"- {item}" for item in (lines or [fallback]))\n\n\ndef compose_heuristic_brief(state: dict[str, Any]) -> str:\n    return "\\n".join(\n        [\n            PREAMBLE,\n            "",\n            render_list(\n                "Objective",\n                [sanitize_text(state.get("objective"))] if sanitize_text(state.get("objective")) else [],\n                "Objective still needs confirmation.",\n            ),\n            "",\n            render_list(\n                "Established context",\n                [sanitize_text(item) for item in (state.get("constraints") or []) if sanitize_text(item)],\n                "No explicit constraints have been captured yet.",\n            ),\n            "",\n            render_list(\n                "Decisions already made",\n                [sanitize_text(item) for item in (state.get("decisions") or []) if sanitize_text(item)],\n                "No firm decisions are recorded yet.",\n            ),\n            "",\n            render_list(\n                "Rejected paths / do not revisit unless necessary",\n                [sanitize_text(item) for item in (state.get("rejected") or []) if sanitize_text(item)],\n                "No rejected paths have been recorded yet.",\n            ),\n            "",\n            render_list(\n                "Open questions",\n                [sanitize_text(item) for item in (state.get("open_questions", state.get("openQuestions", [])) or []) if sanitize_text(item)],\n                "No open questions are active right now.",\n            ),\n            "",\n            render_list(\n                "Where we left off",\n                [sanitize_text(state.get("next_step", state.get("nextStep")))] if sanitize_text(state.get("next_step", state.get("nextStep"))) else [],\n                "Resume from the latest visible turn.",\n            ),\n            "",\n            "Continue from this exact state.",\n            "My next request:",\n            "- ",\n        ]\n    )\n\n\ndef build_recent_turns(row: dict[str, Any]) -> str:\n    turns = [f"- {turn.get(\'role\', \'\')}: {turn.get(\'text\', \'\')}" for turn in row.get("transcript_turns", [])]\n    return "\\n".join(turns) if turns else "- No recent turns provided."\n\n\ndef build_baseline_prompt(\n    row: dict[str, Any],\n    capture_rationale: bool,\n    *,\n    transcript_char_budget: int,\n    per_turn_char_limit: int,\n) -> str:\n    del capture_rationale\n    state = row.get("rolling_state") or {}\n    transcript_text = build_transcript_text(\n        row.get("transcript_turns") or [],\n        per_turn_char_limit=per_turn_char_limit,\n        total_char_budget=transcript_char_budget,\n    )\n    return "\\n".join(\n        [\n            PREAMBLE,\n            "",\n            f"Chat title: {row.get(\'title\', \'\')}",\n            f"Provider: {row.get(\'provider\', \'\')}",\n            "",\n            render_list(\n                "Objective",\n                [sanitize_text(state.get("objective"))] if sanitize_text(state.get("objective")) else [],\n                "Objective still needs confirmation.",\n            ),\n            "",\n            render_list(\n                "Established context",\n                [sanitize_text(item) for item in (state.get("constraints") or []) if sanitize_text(item)],\n                "No explicit constraints.",\n            ),\n            "",\n            render_list(\n                "Decisions already made",\n                [sanitize_text(item) for item in (state.get("decisions") or []) if sanitize_text(item)],\n                "No fixed decisions.",\n            ),\n            "",\n            render_list(\n                "Rejected paths / do not revisit unless necessary",\n                [sanitize_text(item) for item in (state.get("rejected") or []) if sanitize_text(item)],\n                "No rejected paths.",\n            ),\n            "",\n            render_list(\n                "Open questions",\n                [sanitize_text(item) for item in (state.get("open_questions") or []) if sanitize_text(item)],\n                "No open questions.",\n            ),\n            "",\n            render_list(\n                "Where we left off",\n                [sanitize_text(state.get("next_step"))] if sanitize_text(state.get("next_step")) else [],\n                "Resume from latest visible turn.",\n            ),\n            "",\n            "Recent turns:",\n            transcript_text if transcript_text else build_recent_turns(row),\n            "",\n            "Write the final continuation brief now.",\n        ]\n    )\n\n\ndef build_strict_template_prompt(\n    row: dict[str, Any],\n    capture_rationale: bool,\n    *,\n    transcript_char_budget: int,\n    per_turn_char_limit: int,\n) -> str:\n    rolling_state = row.get("rolling_state") or {}\n    transcript_text = build_transcript_text(\n        row.get("transcript_turns") or [],\n        per_turn_char_limit=per_turn_char_limit,\n        total_char_budget=transcript_char_budget,\n    )\n    objective = render_template_list(\n        [sanitize_text(rolling_state.get("objective"))] if sanitize_text(rolling_state.get("objective")) else [],\n        "Objective still needs confirmation.",\n    )\n    context_lines = render_template_list(\n        [sanitize_text(item) for item in (rolling_state.get("constraints") or []) if sanitize_text(item)],\n        "No explicit constraints have been captured yet.",\n    )\n    decision_lines = render_template_list(\n        [sanitize_text(item) for item in (rolling_state.get("decisions") or []) if sanitize_text(item)],\n        "No firm decisions are recorded yet.",\n    )\n    rejected_lines = render_template_list(\n        [sanitize_text(item) for item in (rolling_state.get("rejected") or []) if sanitize_text(item)],\n        "No rejected paths have been recorded yet.",\n    )\n    open_question_lines = render_template_list(\n        [sanitize_text(item) for item in (rolling_state.get("open_questions") or []) if sanitize_text(item)],\n        "No open questions are active right now.",\n    )\n    next_step_lines = render_template_list(\n        [sanitize_text(rolling_state.get("next_step"))] if sanitize_text(rolling_state.get("next_step")) else [],\n        "Resume from the latest visible turn.",\n    )\n    instructions = [\n        "Rewrite the supplied conversation state into the exact brief template below.",\n        "Keep every section header exactly as written.",\n        "Do not add any extra headers, titles, dates, explanations, or code fences.",\n        "Use the transcript as the source of truth and the rolling state only as a helper.",\n        "If something is unclear or not explicit in the transcript, leave it out.",\n        "If the supplied content is already concise and faithful, copy it with only minimal edits.",\n        "Return only the final brief text.",\n    ]\n    if capture_rationale:\n        instructions.extend(\n            [\n                "If you show visible reasoning, place it inside <think>...</think>.",\n                "After the closing </think> tag, return only the final continuation brief.",\n            ]\n        )\n\n    return "\\n".join(\n        [\n            *instructions,\n            "",\n            "Source transcript:",\n            f"- title: {row.get(\'title\', \'\')}",\n            f"- provider: {row.get(\'provider\', \'\')}",\n            transcript_text or "No transcript turns were provided.",\n            "",\n            "Rolling state helper:",\n            render_list(\n                "Objective",\n                [sanitize_text(rolling_state.get("objective"))] if sanitize_text(rolling_state.get("objective")) else [],\n                "Objective still needs confirmation.",\n            ),\n            "",\n            render_list(\n                "Established context",\n                [sanitize_text(item) for item in (rolling_state.get("constraints") or []) if sanitize_text(item)],\n                "No explicit constraints have been captured yet.",\n            ),\n            "",\n            render_list(\n                "Decisions already made",\n                [sanitize_text(item) for item in (rolling_state.get("decisions") or []) if sanitize_text(item)],\n                "No firm decisions are recorded yet.",\n            ),\n            "",\n            render_list(\n                "Rejected paths / do not revisit unless necessary",\n                [sanitize_text(item) for item in (rolling_state.get("rejected") or []) if sanitize_text(item)],\n                "No rejected paths have been recorded yet.",\n            ),\n            "",\n            render_list(\n                "Open questions",\n                [sanitize_text(item) for item in (rolling_state.get("open_questions") or []) if sanitize_text(item)],\n                "No open questions are active right now.",\n            ),\n            "",\n            render_list(\n                "Where we left off",\n                [sanitize_text(rolling_state.get("next_step"))] if sanitize_text(rolling_state.get("next_step")) else [],\n                "Resume from the latest visible turn.",\n            ),\n            "",\n            "Return exactly this structure:",\n            "",\n            PREAMBLE,\n            "",\n            "Objective:",\n            objective,\n            "",\n            "Established context:",\n            context_lines,\n            "",\n            "Decisions already made:",\n            decision_lines,\n            "",\n            "Rejected paths / do not revisit unless necessary:",\n            rejected_lines,\n            "",\n            "Open questions:",\n            open_question_lines,\n            "",\n            "Where we left off:",\n            next_step_lines,\n            "",\n            "Continue from this exact state.",\n            "My next request:",\n            "- ",\n        ]\n    )\n\n\ndef build_heuristic_rewrite_prompt(\n    row: dict[str, Any],\n    capture_rationale: bool,\n    *,\n    transcript_char_budget: int,\n    per_turn_char_limit: int,\n) -> str:\n    del capture_rationale\n    draft = compose_heuristic_brief(row.get("rolling_state") or {})\n    transcript_text = build_transcript_text(\n        row.get("transcript_turns") or [],\n        per_turn_char_limit=per_turn_char_limit,\n        total_char_budget=transcript_char_budget,\n    )\n    return "\\n".join(\n        [\n            "You are editing a continuation brief draft.",\n            "Preserve the exact section headers and order.",\n            "Keep facts faithful to the supplied state.",\n            "If the draft is already good, return it unchanged.",\n            "",\n            "Recent turns for style only:",\n            transcript_text if transcript_text else build_recent_turns(row),\n            "",\n            "Draft to revise:",\n            draft,\n        ]\n    )\n\n\ndef build_prompt(\n    row: dict[str, Any],\n    capture_rationale: bool,\n    *,\n    prompt_method: str = DEFAULT_PROMPT_METHOD,\n    transcript_char_budget: int = DEFAULT_TRANSCRIPT_CHAR_BUDGET,\n    per_turn_char_limit: int = DEFAULT_TRANSCRIPT_TURN_CHAR_LIMIT,\n) -> str:\n    if prompt_method == "baseline":\n        return build_baseline_prompt(\n            row,\n            capture_rationale,\n            transcript_char_budget=transcript_char_budget,\n            per_turn_char_limit=per_turn_char_limit,\n        )\n    if prompt_method == "heuristic_rewrite":\n        return build_heuristic_rewrite_prompt(\n            row,\n            capture_rationale,\n            transcript_char_budget=transcript_char_budget,\n            per_turn_char_limit=per_turn_char_limit,\n        )\n    if prompt_method != "strict_template":\n        raise ValueError(f"Unknown prompt method: {prompt_method}")\n    return build_strict_template_prompt(\n        row,\n        capture_rationale,\n        transcript_char_budget=transcript_char_budget,\n        per_turn_char_limit=per_turn_char_limit,\n    )\n\n\ndef sanitize_model_brief(text: Any) -> str:\n    value = str(text or "")\n    value = re.sub(r"^```[a-z]*\\n?", "", value, flags=re.IGNORECASE)\n    value = re.sub(r"\\n```$", "", value, flags=re.IGNORECASE)\n    return value.strip()\n\n\ndef extract_tagged_rationale(text: str) -> tuple[str, str]:\n    chunks: list[str] = []\n\n    def _replace(match: re.Match[str]) -> str:\n        cleaned = sanitize_model_brief(match.group(1))\n        if cleaned:\n            chunks.append(cleaned)\n        return ""\n\n    remaining = THINK_BLOCK_PATTERN.sub(_replace, text)\n    return "\\n\\n".join(chunks).strip(), remaining.strip()\n\n\ndef split_model_output(raw_text: str) -> tuple[str, str]:\n    sanitized_raw = sanitize_model_brief(raw_text).replace("\\r", "")\n    auxiliary_rationale, remaining = extract_tagged_rationale(sanitized_raw)\n    return remaining or sanitized_raw, auxiliary_rationale\n\n\ndef normalize_generated_brief(text: str) -> str:\n    brief = sanitize_model_brief(text).replace("\\r", "")\n    brief = re.sub(r"\\n{3,}", "\\n\\n", brief)\n    brief = re.sub(r"(?m)^-\\s*", "- ", brief)\n    brief = re.sub(\n        rf"^(?:{re.escape(PREAMBLE)}\\s*\\n+){{2,}}",\n        f"{PREAMBLE}\\n\\n",\n        brief,\n        count=1,\n    )\n    for section, pattern in INLINE_SECTION_PATTERNS.items():\n        brief = pattern.sub(lambda match, section=section: f"{section}\\n- {match.group(1).strip()}", brief)\n    for pattern, replacement in SECTION_HEADER_NORMALIZATIONS:\n        brief = pattern.sub(replacement, brief)\n    brief = re.sub(r"(?m)^My next request:\\s*$\\n(?!-)", "My next request:\\n-", brief)\n    brief = re.sub(r"(?m)^My next request:\\n-\\s*$", "My next request:\\n-", brief)\n    return brief.strip()\n\n\ndef has_required_sections(text: str) -> bool:\n    return all(section in text for section in REQUIRED_SECTIONS)\n\n\ndef has_duplicate_or_missing_section_lines(text: str) -> bool:\n    return any(len(pattern.findall(text)) != 1 for pattern in SECTION_LINE_PATTERNS.values())\n\n\ndef has_prompt_echo(text: str) -> bool:\n    return any(marker in text[:1200] for marker in PROMPT_ECHO_MARKERS)\n\n\ndef has_single_next_request_bullet(text: str) -> bool:\n    match = re.search(r"(?ms)^My next request:\\n(.*)\\Z", text)\n    if not match:\n        return False\n    lines = [line.strip() for line in match.group(1).splitlines() if line.strip()]\n    return len(lines) == 1 and (lines[0] == "-" or lines[0].startswith("- "))\n\n\ndef count_present_content_headers(text: str) -> int:\n    return sum(1 for section in CONTENT_SECTION_HEADERS if section in text)\n\n\ndef is_near_empty_brief(text: str) -> bool:\n    return len(re.sub(r"[^A-Za-z0-9]+", "", text)) < 40\n\n\ndef has_both_tail_anchors_missing(text: str) -> bool:\n    return "Where we left off:" not in text and "My next request:" not in text\n\n\ndef looks_like_unusable_scaffolding(text: str) -> bool:\n    meaningful_lines: list[str] = []\n    for raw_line in text.splitlines():\n        line = raw_line.strip()\n        if not line:\n            continue\n        if line in REQUIRED_SECTIONS or line == "-":\n            continue\n        cleaned = _clean_item(line)\n        if not cleaned or cleaned in GENERIC_FALLBACK_LINES:\n            continue\n        meaningful_lines.append(cleaned)\n    return len(meaningful_lines) < 2\n\n\ndef describe_format_issues(text: str) -> list[str]:\n    issues: list[str] = []\n    missing_sections = [section for section in REQUIRED_SECTIONS if section not in text]\n    if missing_sections:\n        preview = ", ".join(missing_sections[:4])\n        if len(missing_sections) > 4:\n            preview = f"{preview}, +{len(missing_sections) - 4} more"\n        issues.append(f"missing sections: {preview}")\n\n    for section, pattern in SECTION_LINE_PATTERNS.items():\n        count = len(pattern.findall(text))\n        if count == 0:\n            issues.append(f"missing section line: {section}")\n        elif count > 1:\n            issues.append(f"duplicate section line: {section} x{count}")\n\n    if not has_single_next_request_bullet(text):\n        issues.append("invalid My next request bullet")\n    return issues\n\n\ndef summarize_issues(issues: list[str], limit: int = 4) -> str:\n    if not issues:\n        return "no issues"\n    summary = "; ".join(issues[:limit])\n    if len(issues) > limit:\n        summary = f"{summary}; +{len(issues) - limit} more"\n    return summary\n\n\ndef is_format_clean(text: str) -> bool:\n    return (\n        has_required_sections(text)\n        and not has_duplicate_or_missing_section_lines(text)\n        and has_single_next_request_bullet(text)\n        and not PLACEHOLDER_BULLET_PATTERN.search(text)\n        and not has_prompt_echo(text)\n    )\n\n\ndef classify_generated_brief(text: str) -> tuple[str, list[str]]:\n    hard_reject_issues: list[str] = []\n    if is_near_empty_brief(text):\n        hard_reject_issues.append("empty or near-empty normalized output")\n    if has_prompt_echo(text):\n        hard_reject_issues.append("prompt echo in opening text")\n    if PLACEHOLDER_BULLET_PATTERN.search(text):\n        hard_reject_issues.append("contains placeholder bullet")\n    if count_present_content_headers(text) < 4:\n        hard_reject_issues.append("section coverage below 4/8 required content headers")\n    if has_both_tail_anchors_missing(text):\n        hard_reject_issues.append("missing both continuation tail anchors")\n    if looks_like_unusable_scaffolding(text):\n        hard_reject_issues.append("obvious unusable scaffolding")\n    if hard_reject_issues:\n        return "hard_reject", hard_reject_issues\n    if is_format_clean(text):\n        return "format_clean", []\n    format_issues = describe_format_issues(text)\n    return "soft_accept", format_issues or ["format drift"]\n\n\ndef rewrite_tags(tags: list[str], model: str, status: str, has_rationale: bool) -> list[str]:\n    filtered = [\n        tag\n        for tag in list(tags or [])\n        if tag\n        not in {\n            "teacher-draft-backfilled",\n            "teacher-draft-saved",\n            "teacher-draft-generated",\n            "teacher-draft-failed",\n            "teacher-draft-format-clean",\n            "teacher-draft-soft-accepted",\n            "teacher-rationale-captured",\n            "local-heuristic",\n        }\n        and not str(tag).startswith("local-model:")\n    ]\n    if status == "hard_reject":\n        filtered.extend(["teacher-draft-failed", f"local-model:{model}"])\n    else:\n        filtered.extend(["teacher-draft-generated", f"local-model:{model}"])\n        if status == "format_clean":\n            filtered.append("teacher-draft-format-clean")\n        elif status == "soft_accept":\n            filtered.append("teacher-draft-soft-accepted")\n    if has_rationale and status != "hard_reject":\n        filtered.append("teacher-rationale-captured")\n    unique: list[str] = []\n    for tag in filtered:\n        if tag not in unique:\n            unique.append(tag)\n    return unique\n\n\ndef apply_generation_metadata(\n    row: dict[str, Any],\n    *,\n    status: str,\n    issues: list[str],\n    model: str,\n    prompt_method: str,\n    duration_ms: int,\n    auxiliary_rationale: str = "",\n    brief: str = "",\n) -> dict[str, Any]:\n    next_row = dict(row)\n    next_row["teacher_generation_status"] = status\n    next_row["teacher_validation_issues"] = list(issues)\n    next_row["teacher_generation_model"] = model\n    next_row["teacher_generation_prompt"] = prompt_method\n    next_row["teacher_generation_duration_ms"] = duration_ms\n    if status == "hard_reject":\n        next_row.pop("teacher_draft_brief", None)\n        next_row.pop("auxiliary_rationale", None)\n        next_row.pop("auxiliary_rationale_format", None)\n    else:\n        next_row["teacher_draft_brief"] = brief\n        if auxiliary_rationale:\n            next_row["auxiliary_rationale"] = auxiliary_rationale\n            next_row["auxiliary_rationale_format"] = "visible_cot"\n        else:\n            next_row.pop("auxiliary_rationale", None)\n            next_row.pop("auxiliary_rationale_format", None)\n    next_row["tags"] = rewrite_tags(list(row.get("tags", [])), model, status, bool(auxiliary_rationale))\n    return next_row\n\n\ndef count_rows_with_status(rows: list[dict[str, Any]], status: str) -> int:\n    return sum(1 for row in rows if row.get("teacher_generation_status") == status)\n\n\ndef count_rows_with_tag(rows: list[dict[str, Any]], tag: str) -> int:\n    return sum(1 for row in rows if tag in list(row.get("tags", [])))\n\n\ndef build_generation_attempts(requested_max_new_tokens: int) -> list[dict[str, int]]:\n    raw_budgets = [\n        DEFAULT_TRANSCRIPT_CHAR_BUDGET,\n        min(DEFAULT_TRANSCRIPT_CHAR_BUDGET, 10000),\n        min(DEFAULT_TRANSCRIPT_CHAR_BUDGET, 7000),\n        min(DEFAULT_TRANSCRIPT_CHAR_BUDGET, 5000),\n    ]\n    raw_token_caps = [\n        requested_max_new_tokens,\n        min(requested_max_new_tokens, 320),\n        min(requested_max_new_tokens, 256),\n        min(requested_max_new_tokens, 192),\n    ]\n    attempts: list[dict[str, int]] = []\n    for transcript_char_budget, max_new_tokens in zip(raw_budgets, raw_token_caps):\n        attempt = {\n            "transcript_char_budget": transcript_char_budget,\n            "max_new_tokens": max_new_tokens,\n        }\n        if attempt not in attempts:\n            attempts.append(attempt)\n    return attempts\n\n\ndef is_cuda_oom_error(exc: BaseException) -> bool:\n    message = str(exc).lower()\n    return "out of memory" in message and ("cuda" in message or "cublas" in message)\n\n\ndef clear_torch_memory(device: torch.device | None) -> None:\n    gc.collect()\n    if not device or device.type != "cuda" or not torch.cuda.is_available():\n        return\n    try:\n        torch.cuda.empty_cache()\n    except RuntimeError:\n        pass\n    try:\n        torch.cuda.ipc_collect()\n    except RuntimeError:\n        pass\n\n\ndef detect_single_device() -> torch.device:\n    if not torch.cuda.is_available():\n        raise RuntimeError("This notebook requires a CUDA GPU.")\n    return torch.device("cuda")\n\n\ndef resolve_model_load_settings(device_index: int = 0) -> dict[str, Any]:\n    major, minor = torch.cuda.get_device_capability(device_index)\n    capability = (major, minor)\n    if capability < (7, 0):\n        return {\n            "capability": capability,\n            "torch_dtype": torch.float16,\n            "attn_implementation": "eager",\n            "compatibility_mode": "legacy_cuda",\n        }\n    return {\n        "capability": capability,\n        "torch_dtype": torch.float16,\n        "attn_implementation": "sdpa",\n        "compatibility_mode": "default",\n    }\n\n\ndef list_input_json_candidates(root: str = "/kaggle/input") -> list[str]:\n    input_root = Path(root)\n    if not input_root.exists():\n        return []\n    return [str(path) for path in sorted(input_root.rglob("*.json"))]\n\n\ndef resolve_explicit_input_path(explicit_input_path: str) -> str:\n    candidate = Path(explicit_input_path)\n    if candidate.is_file():\n        return str(candidate)\n    if candidate.is_dir():\n        json_candidates = sorted(candidate.rglob("*.json"))\n        if len(json_candidates) == 1:\n            return str(json_candidates[0])\n        for preferred_name in (\n            "teacher_input_rows.json",\n            "live_extension_review_queue_parallel_assistant_repaired_v11.json",\n        ):\n            for json_candidate in json_candidates:\n                if json_candidate.name == preferred_name:\n                    return str(json_candidate)\n        preview = "\\n".join(str(path) for path in json_candidates) if json_candidates else "(no JSON files found)"\n        raise FileNotFoundError(\n            f"Explicit input directory does not resolve to a single JSON file: {candidate}\\nCandidates:\\n{preview}"\n        )\n    raise FileNotFoundError(f"Explicit input path does not exist: {candidate}")\n\n\ndef resolve_known_input_path(run_dataset: str) -> str:\n    dataset_key = run_dataset.strip().lower()\n    if dataset_key not in KNOWN_INPUT_PATHS:\n        raise ValueError(f"Unknown RUN_DATASET value: {run_dataset}")\n    candidate = Path(KNOWN_INPUT_PATHS[dataset_key])\n    if candidate.exists():\n        return str(candidate)\n    available = list_input_json_candidates()\n    preview = "\\n".join(available) if available else "(no JSON files found under /kaggle/input)"\n    raise FileNotFoundError(\n        f"Expected input JSON is missing for RUN_DATASET={run_dataset}: {candidate}\\nAvailable candidates:\\n{preview}"\n    )\n\n\ndef resolve_input_path(run_dataset: str, explicit_input_path: str = "") -> str:\n    explicit_value = explicit_input_path.strip()\n    if explicit_value:\n        return resolve_explicit_input_path(explicit_value)\n    return resolve_known_input_path(run_dataset)\n\n\ndef load_json_rows(path: str) -> list[dict[str, Any]]:\n    data = read_json(path)\n    if not isinstance(data, list):\n        raise RuntimeError(f"{path} does not contain a JSON array.")\n    return data\n\n\ndef select_rows(rows: list[dict[str, Any]], limit: int = 0) -> list[dict[str, Any]]:\n    return rows[:limit] if limit > 0 else list(rows)\n\n\ndef select_split_rows(rows: list[dict[str, Any]], split: str, *, require_gold: bool = False) -> list[dict[str, Any]]:\n    selected = [row for row in rows if str(row.get("split", "")).strip() == split]\n    if require_gold:\n        selected = [row for row in selected if str(row.get("gold_brief", "")).strip()]\n    return selected\n\n\ndef transcript_length_bucket(row: dict[str, Any]) -> str:\n    turn_count = len(row.get("transcript_turns") or [])\n    if turn_count <= 3:\n        return "short"\n    if turn_count <= 9:\n        return "medium"\n    return "long"\n\n\ndef _sorted_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    return sorted(rows, key=lambda row: (str(row.get("conversation_id", "")), str(row.get("case_id", ""))))\n\n\ndef build_stage1_calibration_slice(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    train_gold_rows = _sorted_rows(select_split_rows(rows, "train", require_gold=True))\n    bucket_targets = {"short": 4, "medium": 4, "long": 4}\n    selected_by_bucket: dict[str, list[dict[str, Any]]] = {}\n\n    for bucket_name, target_count in bucket_targets.items():\n        bucket_rows = [row for row in train_gold_rows if transcript_length_bucket(row) == bucket_name]\n        if len(bucket_rows) < target_count:\n            raise RuntimeError(f"Not enough gold-backed train rows for bucket={bucket_name}: need {target_count}, found {len(bucket_rows)}")\n        selected_by_bucket[bucket_name] = bucket_rows[:target_count]\n\n    selected_case_ids = {str(row.get("case_id", "")) for bucket_rows in selected_by_bucket.values() for row in bucket_rows}\n    selected_claude_count = sum(\n        1 for bucket_rows in selected_by_bucket.values() for row in bucket_rows if str(row.get("provider", "")) == "claude"\n    )\n    if selected_claude_count < 2:\n        claude_candidates = [\n            row\n            for row in train_gold_rows\n            if str(row.get("provider", "")) == "claude" and str(row.get("case_id", "")) not in selected_case_ids\n        ]\n        for candidate in claude_candidates:\n            bucket_name = transcript_length_bucket(candidate)\n            replacement_pool = [row for row in reversed(selected_by_bucket[bucket_name]) if str(row.get("provider", "")) != "claude"]\n            if not replacement_pool:\n                continue\n            replacement = replacement_pool[0]\n            replace_index = selected_by_bucket[bucket_name].index(replacement)\n            selected_by_bucket[bucket_name][replace_index] = candidate\n            selected_case_ids.remove(str(replacement.get("case_id", "")))\n            selected_case_ids.add(str(candidate.get("case_id", "")))\n            selected_claude_count += 1\n            if selected_claude_count >= 2:\n                break\n\n    selected_rows: list[dict[str, Any]] = []\n    for bucket_name in ("short", "medium", "long"):\n        selected_rows.extend(_sorted_rows(selected_by_bucket[bucket_name]))\n    return selected_rows\n\n\ndef derive_failure_debug_path(workdir: str, run_mode: str) -> str:\n    return str(Path(workdir) / f"{run_mode}.failures.ndjson")\n\n\ndef derive_soft_accept_review_path(workdir: str, run_mode: str) -> str:\n    return str(Path(workdir) / f"{run_mode}.soft_accept_review.json")\n\n\ndef derive_stage_summary_path(workdir: str, stage_name: str) -> str:\n    return str(Path(workdir) / stage_name / "stage_summary.json")\n\n\ndef arm_paths(base_workdir: str, stage_name: str, arm_name: str) -> dict[str, str]:\n    arm_dir = Path(base_workdir) / stage_name / slugify(arm_name)\n    return {\n        "workdir": str(arm_dir),\n        "output": str(arm_dir / "qwen4b_teacher_drafts_merged.json"),\n        "summary": str(arm_dir / "qwen4b_teacher_drafts_merged.summary.json"),\n    }\n\n\ndef append_failure_debug_record(debug_path: str, payload: dict[str, Any]) -> None:\n    ensure_parent(debug_path)\n    with Path(debug_path).open("a", encoding="utf-8") as handle:\n        handle.write(json.dumps(payload, ensure_ascii=False) + "\\n")\n\n\ndef reset_run_artifacts(*paths: str) -> None:\n    for path in paths:\n        if path and Path(path).exists():\n            Path(path).unlink()\n\n\ndef load_resume_rows(output_path: str, expected_rows: int) -> list[dict[str, Any]]:\n    path = Path(output_path)\n    if not path.exists():\n        return []\n    loaded = json.loads(path.read_text(encoding="utf-8"))\n    if not isinstance(loaded, list):\n        raise RuntimeError("Existing output file does not contain a JSON array.")\n    if len(loaded) > expected_rows:\n        raise RuntimeError("Existing output file has more rows than the selected run.")\n    return loaded\n\n\ndef validate_resume_summary(\n    summary_output_path: str | None,\n    *,\n    input_path: str,\n    model: str,\n    prompt_method: str,\n    run_mode: str,\n    requested_rows: int,\n) -> None:\n    if not summary_output_path:\n        return\n    summary_path = Path(summary_output_path)\n    if not summary_path.exists():\n        return\n    summary = read_json(summary_path)\n    checks = {\n        "inputPath": input_path,\n        "model": model,\n        "promptMethod": prompt_method,\n        "runMode": run_mode,\n        "requestedRows": requested_rows,\n    }\n    mismatches = [\n        f"{key}={summary.get(key)!r} expected {value!r}"\n        for key, value in checks.items()\n        if summary.get(key) != value\n    ]\n    if mismatches:\n        raise RuntimeError("Resume refused because prior summary does not match this run: " + "; ".join(mismatches))\n\n\ndef build_soft_accept_review_entries(rows: list[dict[str, Any]], limit: int = 0) -> list[dict[str, Any]]:\n    soft_rows = [row for row in rows if row.get("teacher_generation_status") == "soft_accept"]\n    entries = [\n        {\n            "case_id": row.get("case_id", ""),\n            "conversation_id": row.get("conversation_id", ""),\n            "title": row.get("title", ""),\n            "provider": row.get("provider", ""),\n            "split": row.get("split", ""),\n            "transcriptTurnCount": len(row.get("transcript_turns") or []),\n            "validationIssues": list(row.get("teacher_validation_issues", [])),\n            "teacherDraftBrief": row.get("teacher_draft_brief", ""),\n        }\n        for row in soft_rows\n    ]\n    if limit > 0:\n        return entries[:limit]\n    return entries\n\n\ndef write_soft_accept_review_queue(review_path: str, rows: list[dict[str, Any]]) -> str:\n    return write_json(review_path, build_soft_accept_review_entries(rows))\n\n\ndef read_failure_records(debug_path: str, limit: int = 5) -> list[dict[str, Any]]:\n    path = Path(debug_path)\n    if not path.exists():\n        return []\n    records: list[dict[str, Any]] = []\n    with path.open("r", encoding="utf-8") as handle:\n        for line in handle:\n            if len(records) >= limit:\n                break\n            line = line.strip()\n            if not line:\n                continue\n            records.append(json.loads(line))\n    return records\n\n\ndef read_review_queue(review_path: str, limit: int = 5) -> list[dict[str, Any]]:\n    path = Path(review_path)\n    if not path.exists():\n        return []\n    records = read_json(path)\n    if limit <= 0:\n        return list(records)\n    return list(records)[:limit]\n\n\ndef build_summary(\n    *,\n    model: str,\n    prompt_method: str,\n    input_path: str,\n    run_mode: str,\n    requested_rows: int,\n    completed_rows: int,\n    generated_rows: list[dict[str, Any]],\n    failures: list[dict[str, str]],\n    capture_rationale: bool,\n    started_at: float,\n    resumed_rows: int,\n    status: str,\n    failure_debug_path: str,\n    soft_accept_review_path: str,\n) -> dict[str, Any]:\n    format_clean_rows = count_rows_with_status(generated_rows, "format_clean")\n    soft_accept_rows = count_rows_with_status(generated_rows, "soft_accept")\n    hard_reject_rows = count_rows_with_status(generated_rows, "hard_reject")\n    usable_rows = format_clean_rows + soft_accept_rows\n    reason_counts: Counter[str] = Counter()\n    durations: list[int] = []\n    for row in generated_rows:\n        for issue in row.get("teacher_validation_issues", []):\n            if row.get("teacher_generation_status") == "hard_reject":\n                reason_counts[str(issue)] += 1\n        duration_value = row.get("teacher_generation_duration_ms")\n        if isinstance(duration_value, (int, float)):\n            durations.append(int(duration_value))\n\n    gold_metrics = aggregate_gold_metrics(generated_rows)\n    shard_complete = completed_rows == requested_rows\n    return {\n        "generatedAt": time.strftime("%Y-%m-%dT%H:%M:%S%z"),\n        "durationMs": round((time.time() - started_at) * 1000),\n        "medianRuntimePerRowMs": int(median(durations)) if durations else 0,\n        "model": model,\n        "promptMethod": prompt_method,\n        "inputPath": input_path,\n        "runMode": run_mode,\n        "inputRows": requested_rows,\n        "requestedRows": requested_rows,\n        "completedRows": completed_rows,\n        "mergedRows": completed_rows,\n        "generatedRows": usable_rows,\n        "formatCleanRows": format_clean_rows,\n        "softAcceptedRows": soft_accept_rows,\n        "hardRejectRows": hard_reject_rows,\n        "failedRows": hard_reject_rows,\n        "usableYield": round(safe_ratio(usable_rows, requested_rows), 4),\n        "hardRejectRate": round(safe_ratio(hard_reject_rows, requested_rows), 4),\n        "promptEchoHardRejects": reason_counts.get("prompt echo in opening text", 0),\n        "hardRejectReasonCounts": dict(sorted(reason_counts.items())),\n        "captureRationale": capture_rationale,\n        "rationaleCapturedRows": sum(1 for row in generated_rows if str(row.get("auxiliary_rationale", "")).strip()),\n        "failures": failures,\n        "goldBackedMetrics": gold_metrics,\n        "numShards": 1,\n        "completeShards": 1 if shard_complete else 0,\n        "shards": [\n            {\n                "shardIndex": 0,\n                "path": "single-run",\n                "expectedRows": requested_rows,\n                "mergedRows": completed_rows,\n                "complete": shard_complete,\n            }\n        ],\n        "shardIndex": 0,\n        "resumedRows": resumed_rows,\n        "status": status,\n        "failureDebugPath": failure_debug_path,\n        "softAcceptReviewPath": soft_accept_review_path,\n    }\n\n\ndef write_checkpoint(\n    *,\n    output_path: str,\n    summary_output: str | None,\n    generated_rows: list[dict[str, Any]],\n    summary: dict[str, Any],\n) -> None:\n    write_json(output_path, generated_rows)\n    if summary_output:\n        write_json(summary_output, summary)\n    write_soft_accept_review_queue(summary["softAcceptReviewPath"], generated_rows)\n\n\nclass HFTeacherGenerator:\n    def __init__(self, model_id: str, temperature: float, max_new_tokens: int) -> None:\n        self.model_id = model_id\n        self.device = detect_single_device()\n        self.model_load_settings = resolve_model_load_settings(0)\n        self.cuda_capability = self.model_load_settings["capability"]\n        self.attn_implementation = self.model_load_settings["attn_implementation"]\n        self.torch_dtype = self.model_load_settings["torch_dtype"]\n        self.compatibility_mode = self.model_load_settings["compatibility_mode"]\n        self.temperature = temperature\n        self.max_new_tokens = max_new_tokens\n        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, use_fast=False)\n        if self.tokenizer.pad_token is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        model_kwargs: dict[str, Any] = {\n            "trust_remote_code": True,\n            "dtype": self.torch_dtype,\n            "attn_implementation": self.attn_implementation,\n        }\n        try:\n            self.model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)\n        except TypeError:\n            model_kwargs.pop("attn_implementation", None)\n            self.model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)\n        self.model.to(self.device)\n        self.model.eval()\n\n    def generate(self, prompt: str, capture_rationale: bool, *, max_new_tokens: int | None = None) -> str:\n        messages = [{"role": "user", "content": prompt}]\n        kwargs: dict[str, Any] = {\n            "add_generation_prompt": True,\n            "tokenize": True,\n            "return_dict": True,\n            "return_tensors": "pt",\n        }\n        if hasattr(self.tokenizer, "apply_chat_template") and self.tokenizer.chat_template:\n            if "qwen" in self.model_id.lower() and not capture_rationale:\n                kwargs["enable_thinking"] = False\n            try:\n                inputs = self.tokenizer.apply_chat_template(messages, **kwargs)\n            except TypeError:\n                kwargs.pop("enable_thinking", None)\n                inputs = self.tokenizer.apply_chat_template(messages, **kwargs)\n        else:\n            inputs = self.tokenizer(prompt, return_tensors="pt")\n\n        inputs = inputs.to(self.device)\n        generated = None\n        new_tokens = None\n        try:\n            with torch.inference_mode():\n                generated = self.model.generate(\n                    **inputs,\n                    do_sample=self.temperature > 0,\n                    temperature=max(self.temperature, 1e-5),\n                    max_new_tokens=max_new_tokens or self.max_new_tokens,\n                    pad_token_id=self.tokenizer.pad_token_id,\n                    eos_token_id=self.tokenizer.eos_token_id,\n                )\n            new_tokens = generated[:, inputs["input_ids"].shape[1] :]\n            return self.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]\n        finally:\n            del inputs\n            if new_tokens is not None:\n                del new_tokens\n            if generated is not None:\n                del generated\n            clear_torch_memory(self.device)\n\n    def close(self) -> None:\n        if hasattr(self, "model"):\n            del self.model\n        if hasattr(self, "tokenizer"):\n            del self.tokenizer\n        clear_torch_memory(self.device)\n\n\ndef _candidate_sort_key(candidate: dict[str, Any]) -> tuple[int, int, int]:\n    return (\n        STATUS_PRIORITY[candidate["status"]],\n        -len(candidate["issues"]),\n        len(candidate["brief"]),\n    )\n\n\ndef _select_better_candidate(left: dict[str, Any] | None, right: dict[str, Any] | None) -> dict[str, Any] | None:\n    if left is None:\n        return right\n    if right is None:\n        return left\n    return max((left, right), key=_candidate_sort_key)\n\n\ndef _evaluate_generation_candidate(\n    *,\n    raw_output: str,\n    prompt: str,\n    attempt_index: int,\n    transcript_char_budget: int,\n    max_new_tokens: int,\n) -> dict[str, Any]:\n    brief_text, auxiliary_rationale = split_model_output(raw_output)\n    brief = normalize_generated_brief(brief_text)\n    status, issues = classify_generated_brief(brief)\n    return {\n        "status": status,\n        "issues": issues,\n        "brief": brief,\n        "auxiliary_rationale": auxiliary_rationale,\n        "raw_output": raw_output,\n        "prompt": prompt,\n        "attempt_index": attempt_index,\n        "transcript_char_budget": transcript_char_budget,\n        "max_new_tokens": max_new_tokens,\n    }\n\n\ndef generate_teacher_draft(\n    *,\n    generator: HFTeacherGenerator,\n    row: dict[str, Any],\n    capture_rationale: bool,\n    prompt_method: str,\n) -> GenerationOutcome:\n    attempts = build_generation_attempts(generator.max_new_tokens)\n    last_oom_error: RuntimeError | None = None\n    attempt_debug_records: list[dict[str, Any]] = []\n    best_candidate: dict[str, Any] | None = None\n    started_at = time.time()\n\n    for attempt_index, attempt in enumerate(attempts, start=1):\n        prompt = build_prompt(\n            row,\n            capture_rationale,\n            prompt_method=prompt_method,\n            transcript_char_budget=attempt["transcript_char_budget"],\n        )\n        try:\n            raw_output = generator.generate(\n                prompt,\n                capture_rationale,\n                max_new_tokens=attempt["max_new_tokens"],\n            )\n            initial_candidate = _evaluate_generation_candidate(\n                raw_output=raw_output,\n                prompt=prompt,\n                attempt_index=attempt_index,\n                transcript_char_budget=attempt["transcript_char_budget"],\n                max_new_tokens=attempt["max_new_tokens"],\n            )\n\n            chosen_candidate = initial_candidate\n            retry_candidate: dict[str, Any] | None = None\n            if initial_candidate["status"] != "format_clean":\n                print(\n                    f"{initial_candidate[\'status\']} for \'{row.get(\'title\', \'\')}\' on attempt {attempt_index}/{len(attempts)}: "\n                    f"{summarize_issues(initial_candidate[\'issues\'])}",\n                    flush=True,\n                )\n                retry_prompt = (\n                    f"{prompt}\\n\\nThe previous answer had structural or content problems. "\n                    "Regenerate the continuation brief with the same facts, no commentary, no template echo, "\n                    "and a filled My next request section."\n                )\n                retry_raw_output = generator.generate(\n                    retry_prompt,\n                    capture_rationale,\n                    max_new_tokens=attempt["max_new_tokens"],\n                )\n                retry_candidate = _evaluate_generation_candidate(\n                    raw_output=retry_raw_output,\n                    prompt=retry_prompt,\n                    attempt_index=attempt_index,\n                    transcript_char_budget=attempt["transcript_char_budget"],\n                    max_new_tokens=attempt["max_new_tokens"],\n                )\n                chosen_candidate = _select_better_candidate(initial_candidate, retry_candidate) or initial_candidate\n\n            best_candidate = _select_better_candidate(best_candidate, chosen_candidate)\n            attempt_debug_records.append(\n                {\n                    "attemptIndex": attempt_index,\n                    "transcriptCharBudget": attempt["transcript_char_budget"],\n                    "maxNewTokens": attempt["max_new_tokens"],\n                    "promptLength": len(prompt),\n                    "promptPreview": trim_debug_text(prompt),\n                    "initialRawOutput": trim_debug_text(raw_output),\n                    "initialNormalizedBrief": trim_debug_text(initial_candidate["brief"]),\n                    "initialStatus": initial_candidate["status"],\n                    "initialIssues": initial_candidate["issues"],\n                    "retryRawOutput": trim_debug_text(retry_candidate["raw_output"]) if retry_candidate else "",\n                    "retryNormalizedBrief": trim_debug_text(retry_candidate["brief"]) if retry_candidate else "",\n                    "retryStatus": retry_candidate["status"] if retry_candidate else "",\n                    "retryIssues": retry_candidate["issues"] if retry_candidate else [],\n                    "chosenStatus": chosen_candidate["status"],\n                    "chosenIssues": chosen_candidate["issues"],\n                }\n            )\n\n            if chosen_candidate["status"] == "format_clean":\n                break\n        except RuntimeError as exc:\n            if not is_cuda_oom_error(exc):\n                raise\n            last_oom_error = exc\n            print(\n                "OOM while generating "\n                f"\'{row.get(\'title\', \'\')}\' on attempt {attempt_index}/{len(attempts)}; "\n                f"retrying with transcript_char_budget={attempt[\'transcript_char_budget\']} "\n                f"and max_new_tokens={attempt[\'max_new_tokens\']}",\n                flush=True,\n            )\n            clear_torch_memory(generator.device)\n\n    if best_candidate is not None:\n        duration_ms = round((time.time() - started_at) * 1000)\n        return GenerationOutcome(\n            brief=best_candidate["brief"],\n            auxiliary_rationale=best_candidate["auxiliary_rationale"],\n            status=best_candidate["status"],\n            issues=list(best_candidate["issues"]),\n            debug_payload={\n                "attempts": attempt_debug_records,\n                "chosenStatus": best_candidate["status"],\n                "chosenIssues": list(best_candidate["issues"]),\n            },\n            duration_ms=duration_ms,\n        )\n\n    if last_oom_error is not None:\n        raise RuntimeError(f"{last_oom_error} after {len(attempts)} memory-recovery attempts.") from last_oom_error\n    raise RuntimeError("Teacher draft generation exhausted all attempts without producing output.")\n\n\ndef run_generation(\n    *,\n    generator: HFTeacherGenerator,\n    rows: list[dict[str, Any]],\n    input_path: str,\n    output_path: str,\n    summary_output: str | None,\n    workdir: str,\n    model: str,\n    prompt_method: str,\n    run_mode: str,\n    checkpoint_every: int,\n    capture_rationale: bool,\n    resume: bool,\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\n    started_at = time.time()\n    selected_rows = list(rows)\n    failure_debug_path = derive_failure_debug_path(workdir, run_mode)\n    soft_accept_review_path = derive_soft_accept_review_path(workdir, run_mode)\n\n    if resume:\n        validate_resume_summary(\n            summary_output,\n            input_path=input_path,\n            model=model,\n            prompt_method=prompt_method,\n            run_mode=run_mode,\n            requested_rows=len(selected_rows),\n        )\n        generated_rows = load_resume_rows(output_path, len(selected_rows))\n        if generated_rows:\n            print(f"Resuming {run_mode} run from row {len(generated_rows) + 1}/{len(selected_rows)}", flush=True)\n    else:\n        reset_run_artifacts(output_path, summary_output or "", failure_debug_path, soft_accept_review_path)\n        generated_rows = []\n\n    failures: list[dict[str, str]] = []\n    resumed_rows = len(generated_rows)\n\n    for index, row in enumerate(selected_rows[resumed_rows:], start=resumed_rows + 1):\n        print(\n            f"Generating {index}/{len(selected_rows)} [{prompt_method}] [{model}]: {row.get(\'title\', \'\')}",\n            flush=True,\n        )\n        try:\n            outcome = generate_teacher_draft(\n                generator=generator,\n                row=row,\n                capture_rationale=capture_rationale,\n                prompt_method=prompt_method,\n            )\n            next_row = apply_generation_metadata(\n                row,\n                status=outcome.status,\n                issues=outcome.issues,\n                model=model,\n                prompt_method=prompt_method,\n                duration_ms=outcome.duration_ms,\n                auxiliary_rationale=outcome.auxiliary_rationale,\n                brief=outcome.brief,\n            )\n            generated_rows.append(next_row)\n            if outcome.status == "hard_reject":\n                failure_record = {\n                    "title": str(row.get("title", "")),\n                    "conversation_id": str(row.get("conversation_id", "")),\n                    "message": summarize_issues(outcome.issues),\n                }\n                failures.append(failure_record)\n                append_failure_debug_record(\n                    failure_debug_path,\n                    {\n                        "title": failure_record["title"],\n                        "conversation_id": failure_record["conversation_id"],\n                        "provider": str(row.get("provider", "")),\n                        "model": model,\n                        "promptMethod": prompt_method,\n                        "runMode": run_mode,\n                        "errorType": "HardReject",\n                        "message": summarize_issues(outcome.issues),\n                        **outcome.debug_payload,\n                    },\n                )\n        except Exception as exc:  # noqa: BLE001\n            issues = [f"runtime/model exception: {type(exc).__name__}", str(exc)]\n            failure_record = {\n                "title": str(row.get("title", "")),\n                "conversation_id": str(row.get("conversation_id", "")),\n                "message": str(exc),\n            }\n            failures.append(failure_record)\n            generated_rows.append(\n                apply_generation_metadata(\n                    row,\n                    status="hard_reject",\n                    issues=issues,\n                    model=model,\n                    prompt_method=prompt_method,\n                    duration_ms=0,\n                )\n            )\n            append_failure_debug_record(\n                failure_debug_path,\n                {\n                    "title": failure_record["title"],\n                    "conversation_id": failure_record["conversation_id"],\n                    "provider": str(row.get("provider", "")),\n                    "model": model,\n                    "promptMethod": prompt_method,\n                    "runMode": run_mode,\n                    "errorType": type(exc).__name__,\n                    "message": str(exc),\n                },\n            )\n\n        if len(generated_rows) % max(checkpoint_every, 1) == 0 or index == len(selected_rows):\n            summary = build_summary(\n                model=model,\n                prompt_method=prompt_method,\n                input_path=input_path,\n                run_mode=run_mode,\n                requested_rows=len(selected_rows),\n                completed_rows=len(generated_rows),\n                generated_rows=generated_rows,\n                failures=failures,\n                capture_rationale=capture_rationale,\n                started_at=started_at,\n                resumed_rows=resumed_rows,\n                status="running" if index < len(selected_rows) else "completed",\n                failure_debug_path=failure_debug_path,\n                soft_accept_review_path=soft_accept_review_path,\n            )\n            write_checkpoint(\n                output_path=output_path,\n                summary_output=summary_output,\n                generated_rows=generated_rows,\n                summary=summary,\n            )\n\n    final_summary = build_summary(\n        model=model,\n        prompt_method=prompt_method,\n        input_path=input_path,\n        run_mode=run_mode,\n        requested_rows=len(selected_rows),\n        completed_rows=len(generated_rows),\n        generated_rows=generated_rows,\n        failures=failures,\n        capture_rationale=capture_rationale,\n        started_at=started_at,\n        resumed_rows=resumed_rows,\n        status="completed",\n        failure_debug_path=failure_debug_path,\n        soft_accept_review_path=soft_accept_review_path,\n    )\n    write_checkpoint(\n        output_path=output_path,\n        summary_output=summary_output,\n        generated_rows=generated_rows,\n        summary=final_summary,\n    )\n    return generated_rows, final_summary\n\n\ndef _arm_ranking_tuple(summary: dict[str, Any]) -> tuple[float, float, float]:\n    metrics = summary.get("goldBackedMetrics", {})\n    return (\n        float(metrics.get("state_fidelity_score", 0.0)),\n        float(metrics.get("next_step_fidelity", 0.0)),\n        -float(metrics.get("rejected_path_reintroduction_rate", 1.0)),\n    )\n\n\ndef _model_ranking_tuple(summary: dict[str, Any]) -> tuple[float, float, float, float]:\n    metrics = summary.get("goldBackedMetrics", {})\n    return (\n        float(metrics.get("state_fidelity_score", 0.0)),\n        float(summary.get("usableYield", 0.0)),\n        -float(metrics.get("hallucination_penalty", 1.0)),\n        -float(summary.get("medianRuntimePerRowMs", 1e12)),\n    )\n\n\ndef stage1_arm_qualifies(summary: dict[str, Any]) -> bool:\n    return (\n        float(summary.get("usableYield", 0.0)) >= 0.80\n        and float(summary.get("hardRejectRate", 1.0)) <= 0.20\n        and int(summary.get("promptEchoHardRejects", 1)) == 0\n    )\n\n\ndef stage2_arm_qualifies(summary: dict[str, Any]) -> bool:\n    metrics = summary.get("goldBackedMetrics", {})\n    return (\n        float(summary.get("usableYield", 0.0)) >= 0.85\n        and float(summary.get("hardRejectRate", 1.0)) <= 0.10\n        and float(metrics.get("next_step_fidelity", 0.0)) >= 0.65\n        and float(metrics.get("rejected_path_reintroduction_rate", 1.0)) <= 0.10\n    )\n\n\ndef stage3_holdout_passes(summary: dict[str, Any], *, validation_state_fidelity: float) -> bool:\n    metrics = summary.get("goldBackedMetrics", {})\n    holdout_state_fidelity = float(metrics.get("state_fidelity_score", 0.0))\n    return (\n        float(summary.get("usableYield", 0.0)) >= 0.80\n        and float(summary.get("hardRejectRate", 1.0)) <= 0.15\n        and abs(holdout_state_fidelity - validation_state_fidelity) <= 0.05\n    )\n\n\ndef _sample_rows_by_bucket(rows: list[dict[str, Any]], count: int) -> list[dict[str, Any]]:\n    if count <= 0 or not rows:\n        return []\n    ordered_rows = _sorted_rows(rows)\n    bucket_rows = {\n        "short": [row for row in ordered_rows if transcript_length_bucket(row) == "short"],\n        "medium": [row for row in ordered_rows if transcript_length_bucket(row) == "medium"],\n        "long": [row for row in ordered_rows if transcript_length_bucket(row) == "long"],\n    }\n    sample: list[dict[str, Any]] = []\n    while len(sample) < count:\n        progress = False\n        for bucket_name in ("short", "medium", "long"):\n            if bucket_rows[bucket_name]:\n                sample.append(bucket_rows[bucket_name].pop(0))\n                progress = True\n                if len(sample) >= count:\n                    break\n        if not progress:\n            break\n    return sample\n\n\ndef build_review_sample(rows: list[dict[str, Any]], count: int) -> list[dict[str, Any]]:\n    sampled = _sample_rows_by_bucket([row for row in rows if row.get("teacher_generation_status") == "soft_accept"], count)\n    return [\n        {\n            "case_id": row.get("case_id", ""),\n            "title": row.get("title", ""),\n            "provider": row.get("provider", ""),\n            "split": row.get("split", ""),\n            "status": row.get("teacher_generation_status", ""),\n            "validationIssues": list(row.get("teacher_validation_issues", [])),\n        }\n        for row in sampled\n    ]\n\n\ndef _run_arm(\n    *,\n    rows: list[dict[str, Any]],\n    input_path: str,\n    base_workdir: str,\n    stage_name: str,\n    arm_name: str,\n    model: str,\n    prompt_method: str,\n    checkpoint_every: int,\n    capture_rationale: bool,\n    temperature: float,\n    max_new_tokens: int,\n) -> dict[str, Any]:\n    paths = arm_paths(base_workdir, stage_name, arm_name)\n    generator = HFTeacherGenerator(model, temperature, max_new_tokens)\n    try:\n        generated_rows, summary = run_generation(\n            generator=generator,\n            rows=rows,\n            input_path=input_path,\n            output_path=paths["output"],\n            summary_output=paths["summary"],\n            workdir=paths["workdir"],\n            model=model,\n            prompt_method=prompt_method,\n            run_mode=stage_name,\n            checkpoint_every=checkpoint_every,\n            capture_rationale=capture_rationale,\n            resume=False,\n        )\n    finally:\n        generator.close()\n    return {\n        "armName": arm_name,\n        "model": model,\n        "promptMethod": prompt_method,\n        "outputPath": paths["output"],\n        "summaryPath": paths["summary"],\n        "summary": summary,\n        "reviewSample": build_review_sample(generated_rows, 6 if stage_name == "stage2_model" else 10),\n    }\n\n\ndef run_stage1_prompt_screen(\n    *,\n    rows: list[dict[str, Any]],\n    input_path: str,\n    base_workdir: str,\n    checkpoint_every: int,\n    capture_rationale: bool,\n    temperature: float,\n    max_new_tokens: int,\n    model: str = DEFAULT_STAGE1_MODEL,\n    prompt_methods: tuple[str, ...] = PROMPT_METHODS,\n) -> dict[str, Any]:\n    calibration_rows = build_stage1_calibration_slice(rows)\n    arm_results = [\n        _run_arm(\n            rows=calibration_rows,\n            input_path=input_path,\n            base_workdir=base_workdir,\n            stage_name="stage1_prompt",\n            arm_name=prompt_method,\n            model=model,\n            prompt_method=prompt_method,\n            checkpoint_every=checkpoint_every,\n            capture_rationale=capture_rationale,\n            temperature=temperature,\n            max_new_tokens=max_new_tokens,\n        )\n        for prompt_method in prompt_methods\n    ]\n    shortlisted = [result for result in arm_results if stage1_arm_qualifies(result["summary"])]\n    winner = max(shortlisted, key=lambda result: _arm_ranking_tuple(result["summary"])) if shortlisted else None\n    stage_summary = {\n        "stage": "stage1_prompt",\n        "calibrationRows": len(calibration_rows),\n        "selectedCaseIds": [row.get("case_id", "") for row in calibration_rows],\n        "selectedTitles": [row.get("title", "") for row in calibration_rows],\n        "shortlistedPromptMethods": [result["promptMethod"] for result in shortlisted],\n        "winnerPromptMethod": winner["promptMethod"] if winner else None,\n        "pass": winner is not None,\n        "results": arm_results,\n    }\n    write_json(derive_stage_summary_path(base_workdir, "stage1_prompt"), stage_summary)\n    return stage_summary\n\n\ndef run_stage2_model_selection(\n    *,\n    rows: list[dict[str, Any]],\n    input_path: str,\n    base_workdir: str,\n    checkpoint_every: int,\n    capture_rationale: bool,\n    temperature: float,\n    max_new_tokens: int,\n    prompt_method: str,\n    models: tuple[str, ...] = DEFAULT_STAGE2_MODELS,\n) -> dict[str, Any]:\n    val_rows = select_split_rows(rows, "val")\n    arm_results = [\n        _run_arm(\n            rows=val_rows,\n            input_path=input_path,\n            base_workdir=base_workdir,\n            stage_name="stage2_model",\n            arm_name=model,\n            model=model,\n            prompt_method=prompt_method,\n            checkpoint_every=checkpoint_every,\n            capture_rationale=capture_rationale,\n            temperature=temperature,\n            max_new_tokens=max_new_tokens,\n        )\n        for model in models\n    ]\n    qualified = [result for result in arm_results if stage2_arm_qualifies(result["summary"])]\n    ranked_results = sorted(arm_results, key=lambda result: _model_ranking_tuple(result["summary"]), reverse=True)\n    winner = max(qualified, key=lambda result: _model_ranking_tuple(result["summary"])) if qualified else None\n    runner_up = next((result for result in ranked_results if winner is not None and result["model"] != winner["model"]), None)\n    stage_summary = {\n        "stage": "stage2_model",\n        "validationRows": len(val_rows),\n        "goldBackedValidationRows": sum(1 for row in val_rows if str(row.get("gold_brief", "")).strip()),\n        "promptMethod": prompt_method,\n        "qualifiedModels": [result["model"] for result in qualified],\n        "winnerModel": winner["model"] if winner else None,\n        "runnerUpModel": runner_up["model"] if runner_up else None,\n        "pass": winner is not None,\n        "results": arm_results,\n    }\n    write_json(derive_stage_summary_path(base_workdir, "stage2_model"), stage_summary)\n    return stage_summary\n\n\ndef run_stage3_holdout_check(\n    *,\n    rows: list[dict[str, Any]],\n    input_path: str,\n    base_workdir: str,\n    checkpoint_every: int,\n    capture_rationale: bool,\n    temperature: float,\n    max_new_tokens: int,\n    prompt_method: str,\n    winner_model: str,\n    runner_up_model: str | None,\n    validation_state_fidelity: float,\n) -> dict[str, Any]:\n    test_rows = select_split_rows(rows, "test")\n    initial_result = _run_arm(\n        rows=test_rows,\n        input_path=input_path,\n        base_workdir=base_workdir,\n        stage_name="stage3_holdout",\n        arm_name=winner_model,\n        model=winner_model,\n        prompt_method=prompt_method,\n        checkpoint_every=checkpoint_every,\n        capture_rationale=capture_rationale,\n        temperature=temperature,\n        max_new_tokens=max_new_tokens,\n    )\n    initial_pass = stage3_holdout_passes(initial_result["summary"], validation_state_fidelity=validation_state_fidelity)\n    final_result = initial_result\n    rerun_result: dict[str, Any] | None = None\n    if not initial_pass and runner_up_model and runner_up_model != winner_model:\n        rerun_result = _run_arm(\n            rows=test_rows,\n            input_path=input_path,\n            base_workdir=base_workdir,\n            stage_name="stage3_holdout_rerun",\n            arm_name=runner_up_model,\n            model=runner_up_model,\n            prompt_method=prompt_method,\n            checkpoint_every=checkpoint_every,\n            capture_rationale=capture_rationale,\n            temperature=temperature,\n            max_new_tokens=max_new_tokens,\n        )\n        if stage3_holdout_passes(rerun_result["summary"], validation_state_fidelity=validation_state_fidelity):\n            final_result = rerun_result\n\n    stage_summary = {\n        "stage": "stage3_holdout",\n        "testRows": len(test_rows),\n        "goldBackedTestRows": sum(1 for row in test_rows if str(row.get("gold_brief", "")).strip()),\n        "promptMethod": prompt_method,\n        "initialModel": winner_model,\n        "runnerUpModel": runner_up_model,\n        "chosenModel": final_result["model"],\n        "pass": stage3_holdout_passes(final_result["summary"], validation_state_fidelity=validation_state_fidelity),\n        "validationReferenceStateFidelity": validation_state_fidelity,\n        "initialResult": initial_result,\n        "rerunResult": rerun_result,\n    }\n    write_json(derive_stage_summary_path(base_workdir, "stage3_holdout"), stage_summary)\n    return stage_summary\n\n'
RUNTIME_PATH = Path("/kaggle/working/teacher_runtime.py")
RUNTIME_PATH.write_text(EMBEDDED_RUNTIME_SOURCE, encoding="utf-8")

if "teacher_runtime" in sys.modules:
    del sys.modules["teacher_runtime"]

import torch
spec = importlib.util.spec_from_file_location("teacher_runtime", RUNTIME_PATH)
teacher_runtime = importlib.util.module_from_spec(spec)
sys.modules["teacher_runtime"] = teacher_runtime
spec.loader.exec_module(teacher_runtime)

DEFAULT_MODEL = teacher_runtime.DEFAULT_MODEL
DEFAULT_STAGE1_MODEL = teacher_runtime.DEFAULT_STAGE1_MODEL
DEFAULT_STAGE2_MODELS = teacher_runtime.DEFAULT_STAGE2_MODELS
PROMPT_METHODS = teacher_runtime.PROMPT_METHODS
HFTeacherGenerator = teacher_runtime.HFTeacherGenerator
KNOWN_INPUT_PATHS = teacher_runtime.KNOWN_INPUT_PATHS
build_stage1_calibration_slice = teacher_runtime.build_stage1_calibration_slice
build_review_sample = teacher_runtime.build_review_sample
list_input_json_candidates = teacher_runtime.list_input_json_candidates
load_json_rows = teacher_runtime.load_json_rows
read_failure_records = teacher_runtime.read_failure_records
read_review_queue = teacher_runtime.read_review_queue
resolve_input_path = teacher_runtime.resolve_input_path
run_generation = teacher_runtime.run_generation
run_stage1_prompt_screen = teacher_runtime.run_stage1_prompt_screen
run_stage2_model_selection = teacher_runtime.run_stage2_model_selection
run_stage3_holdout_check = teacher_runtime.run_stage3_holdout_check
select_split_rows = teacher_runtime.select_split_rows
write_json = teacher_runtime.write_json

print(f"Embedded runtime written to: {RUNTIME_PATH}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Torch CUDA build: {torch.version.cuda}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    print(f"Primary GPU: {torch.cuda.get_device_name(0)}")
    print(f"Primary GPU capability: {capability[0]}.{capability[1]}")



In [ ]:
RUN_DATASET = "full"  # full | repair
INPUT_PATH_OVERRIDE = "/kaggle/input/datasets/sujendragharat/qwen-4b-teacher-2gpu-inputs-new"
STAGE1_MODEL = DEFAULT_STAGE1_MODEL
STAGE2_MODELS = DEFAULT_STAGE2_MODELS
PROMPT_ARMS = PROMPT_METHODS
CHECKPOINT_EVERY = 1
CAPTURE_RATIONALE = False
MAX_NEW_TOKENS = 420
TEMPERATURE = 0.2
ALLOW_FULL_RUN = False

NOTEBOOK_WORKDIR = "/kaggle/working/qwen_teacher_t4_notebook"
STAGE1_SUMMARY_PATH = f"{NOTEBOOK_WORKDIR}/stage1_prompt/stage_summary.json"
STAGE2_SUMMARY_PATH = f"{NOTEBOOK_WORKDIR}/stage2_model/stage_summary.json"
STAGE3_SUMMARY_PATH = f"{NOTEBOOK_WORKDIR}/stage3_holdout/stage_summary.json"
FULL_OUTPUT_PATH = "/kaggle/working/qwen4b_teacher_drafts_merged.json"
FULL_SUMMARY_OUTPUT = "/kaggle/working/qwen4b_teacher_drafts_merged.summary.json"

print(json.dumps({
    "RUN_DATASET": RUN_DATASET,
    "INPUT_PATH_OVERRIDE": INPUT_PATH_OVERRIDE,
    "STAGE1_MODEL": STAGE1_MODEL,
    "STAGE2_MODELS": STAGE2_MODELS,
    "PROMPT_ARMS": PROMPT_ARMS,
    "CHECKPOINT_EVERY": CHECKPOINT_EVERY,
    "CAPTURE_RATIONALE": CAPTURE_RATIONALE,
    "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
    "TEMPERATURE": TEMPERATURE,
    "ALLOW_FULL_RUN": ALLOW_FULL_RUN,
    "NOTEBOOK_WORKDIR": NOTEBOOK_WORKDIR,
    "FULL_OUTPUT_PATH": FULL_OUTPUT_PATH,
    "FULL_SUMMARY_OUTPUT": FULL_SUMMARY_OUTPUT,
}, indent=2))



In [ ]:
from collections import Counter

INPUT_PATH = resolve_input_path(RUN_DATASET, INPUT_PATH_OVERRIDE)
ALL_ROWS = load_json_rows(INPUT_PATH)
TRAIN_ROWS = select_split_rows(ALL_ROWS, "train")
VAL_ROWS = select_split_rows(ALL_ROWS, "val")
TEST_ROWS = select_split_rows(ALL_ROWS, "test")
STAGE1_ROWS = build_stage1_calibration_slice(ALL_ROWS)

print(f"Resolved INPUT_PATH: {INPUT_PATH}")
print("Known mounted paths:")
print(json.dumps(KNOWN_INPUT_PATHS, indent=2))
print(f"Rows loaded: {len(ALL_ROWS)}")
print("Split counts:")
print(json.dumps({
    "train": len(TRAIN_ROWS),
    "val": len(VAL_ROWS),
    "test": len(TEST_ROWS),
}, indent=2))
print("Gold-backed counts by split:")
print(json.dumps(dict(Counter(row.get("split") for row in ALL_ROWS if str(row.get("gold_brief", "")).strip())), indent=2))
print("Stage 1 calibration titles:")
for row in STAGE1_ROWS:
    print("-", row.get("title", ""), f"[{row.get('provider', '')}]", f"({len(row.get('transcript_turns', []))} turns)")
print("Visible JSON candidates under /kaggle/input:")
for path in list_input_json_candidates():
    print("-", path)



In [ ]:
STAGE1_SUMMARY = run_stage1_prompt_screen(
    rows=ALL_ROWS,
    input_path=INPUT_PATH,
    base_workdir=NOTEBOOK_WORKDIR,
    checkpoint_every=CHECKPOINT_EVERY,
    capture_rationale=CAPTURE_RATIONALE,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    model=STAGE1_MODEL,
    prompt_methods=PROMPT_ARMS,
)
print(json.dumps(STAGE1_SUMMARY, indent=2))



In [ ]:
print(json.dumps(STAGE1_SUMMARY, indent=2))
if not STAGE1_SUMMARY["pass"]:
    raise RuntimeError("Stage 1 did not produce a qualified prompt winner.")

WINNING_PROMPT_METHOD = STAGE1_SUMMARY["winnerPromptMethod"]
print(f"Winning prompt method: {WINNING_PROMPT_METHOD}")
print("Per-arm usable yield / hard reject rate:")
for result in STAGE1_SUMMARY["results"]:
    summary = result["summary"]
    print(
        result["promptMethod"],
        {
            "usableYield": summary["usableYield"],
            "hardRejectRate": summary["hardRejectRate"],
            "state_fidelity_score": summary.get("goldBackedMetrics", {}).get("state_fidelity_score", 0.0),
            "next_step_fidelity": summary.get("goldBackedMetrics", {}).get("next_step_fidelity", 0.0),
        },
    )
    print("Sample hard rejects:")
    for record in read_failure_records(summary["failureDebugPath"], limit=2):
        print(json.dumps(record, indent=2)[:3000])
        print("-" * 80)
    print("Sample soft accepts:")
    for record in read_review_queue(summary["softAcceptReviewPath"], limit=2):
        print(json.dumps(record, indent=2)[:3000])
        print("-" * 80)



In [ ]:
STAGE2_SUMMARY = run_stage2_model_selection(
    rows=ALL_ROWS,
    input_path=INPUT_PATH,
    base_workdir=NOTEBOOK_WORKDIR,
    checkpoint_every=CHECKPOINT_EVERY,
    capture_rationale=CAPTURE_RATIONALE,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    prompt_method=WINNING_PROMPT_METHOD,
    models=STAGE2_MODELS,
)
print(json.dumps(STAGE2_SUMMARY, indent=2))



In [ ]:
print(json.dumps(STAGE2_SUMMARY, indent=2))
if not STAGE2_SUMMARY["pass"]:
    raise RuntimeError("Stage 2 did not produce a qualified model winner.")

WINNING_MODEL = STAGE2_SUMMARY["winnerModel"]
RUNNER_UP_MODEL = STAGE2_SUMMARY["runnerUpModel"]
VALIDATION_REFERENCE_STATE_FIDELITY = next(
    result["summary"].get("goldBackedMetrics", {}).get("state_fidelity_score", 0.0)
    for result in STAGE2_SUMMARY["results"]
    if result["model"] == WINNING_MODEL
)
print(f"Winning model: {WINNING_MODEL}")
print(f"Runner-up model: {RUNNER_UP_MODEL}")
print(f"Validation reference state_fidelity_score: {VALIDATION_REFERENCE_STATE_FIDELITY}")
for result in STAGE2_SUMMARY["results"]:
    summary = result["summary"]
    print(
        result["model"],
        {
            "usableYield": summary["usableYield"],
            "hardRejectRate": summary["hardRejectRate"],
            "state_fidelity_score": summary.get("goldBackedMetrics", {}).get("state_fidelity_score", 0.0),
            "next_step_fidelity": summary.get("goldBackedMetrics", {}).get("next_step_fidelity", 0.0),
            "hallucination_penalty": summary.get("goldBackedMetrics", {}).get("hallucination_penalty", 0.0),
            "medianRuntimePerRowMs": summary.get("medianRuntimePerRowMs", 0),
        },
    )
    print("Sample hard rejects:")
    for record in read_failure_records(summary["failureDebugPath"], limit=2):
        print(json.dumps(record, indent=2)[:3000])
        print("-" * 80)
    print("Sample soft accepts:")
    for record in read_review_queue(summary["softAcceptReviewPath"], limit=2):
        print(json.dumps(record, indent=2)[:3000])
        print("-" * 80)



In [ ]:
STAGE3_SUMMARY = run_stage3_holdout_check(
    rows=ALL_ROWS,
    input_path=INPUT_PATH,
    base_workdir=NOTEBOOK_WORKDIR,
    checkpoint_every=CHECKPOINT_EVERY,
    capture_rationale=CAPTURE_RATIONALE,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    prompt_method=WINNING_PROMPT_METHOD,
    winner_model=WINNING_MODEL,
    runner_up_model=RUNNER_UP_MODEL,
    validation_state_fidelity=VALIDATION_REFERENCE_STATE_FIDELITY,
)
print(json.dumps(STAGE3_SUMMARY, indent=2))



In [ ]:
print(json.dumps(STAGE3_SUMMARY, indent=2))
if not STAGE3_SUMMARY["pass"]:
    raise RuntimeError("Stage 3 holdout did not pass. Do not run the full dataset yet.")

CHOSEN_MODEL = STAGE3_SUMMARY["chosenModel"]
print(f"Chosen model for full run: {CHOSEN_MODEL}")
print("Initial holdout hard rejects:")
for record in read_failure_records(STAGE3_SUMMARY["initialResult"]["summary"]["failureDebugPath"], limit=5):
    print(json.dumps(record, indent=2)[:3000])
    print("-" * 80)
if STAGE3_SUMMARY.get("rerunResult"):
    print("Runner-up holdout hard rejects:")
    for record in read_failure_records(STAGE3_SUMMARY["rerunResult"]["summary"]["failureDebugPath"], limit=5):
        print(json.dumps(record, indent=2)[:3000])
        print("-" * 80)



In [ ]:
if not ALLOW_FULL_RUN:
    raise RuntimeError("Set ALLOW_FULL_RUN = True in the config cell after reviewing Stage 3.")
if "STAGE3_SUMMARY" not in globals() or not STAGE3_SUMMARY["pass"]:
    raise RuntimeError("Stage 3 did not pass. Review the holdout outputs before starting the full run.")

FULL_GENERATOR = HFTeacherGenerator(CHOSEN_MODEL, TEMPERATURE, MAX_NEW_TOKENS)
try:
    FULL_GENERATED_ROWS, FULL_SUMMARY = run_generation(
        generator=FULL_GENERATOR,
        rows=ALL_ROWS,
        input_path=INPUT_PATH,
        output_path=FULL_OUTPUT_PATH,
        summary_output=FULL_SUMMARY_OUTPUT,
        workdir=NOTEBOOK_WORKDIR,
        model=CHOSEN_MODEL,
        prompt_method=WINNING_PROMPT_METHOD,
        run_mode="full",
        checkpoint_every=CHECKPOINT_EVERY,
        capture_rationale=CAPTURE_RATIONALE,
        resume=False,
    )
finally:
    FULL_GENERATOR.close()
print(json.dumps(FULL_SUMMARY, indent=2))



In [ ]:
print(json.dumps(FULL_SUMMARY, indent=2))
print("Sample passing rows:")
passing_rows = [row for row in FULL_GENERATED_ROWS if row.get("teacher_generation_status") in {"format_clean", "soft_accept"}]
for row in passing_rows[:3]:
    print("=" * 80)
    print(row.get("title", ""), row.get("teacher_generation_status", ""))
    print(row.get("teacher_draft_brief", "")[:2000])

print("Sample hard reject records:")
for record in read_failure_records(FULL_SUMMARY["failureDebugPath"], limit=5):
    print(json.dumps(record, indent=2)[:3000])
    print("-" * 80)

print("Sample soft accept review entries:")
for record in read_review_queue(FULL_SUMMARY["softAcceptReviewPath"], limit=5):
    print(json.dumps(record, indent=2)[:3000])
    print("-" * 80)

